In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
DATA_DIR = Path("../data/processed")
FEATURE_DIR = DATA_DIR / "features"

movies = pd.read_csv(DATA_DIR / "movies_clean.csv")

genre_df = pd.read_csv(
    FEATURE_DIR / "movie_genre_features.csv",
    index_col=0
)

genre_df.index = genre_df.index.astype(int)

print("Movies:", movies.shape)
print("Genre features:", genre_df.shape)

Movies: (3883, 3)
Genre features: (3883, 18)


In [3]:
movie_details = movies.set_index("MovieID").loc[genre_df.index]

print("Movie and feature alignment:", 
      movie_details.index.equals(genre_df.index))

Movie and feature alignment: True


In [4]:
similarity_matrix = cosine_similarity(genre_df)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (3883, 3883)


In [5]:
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=genre_df.index,
    columns=genre_df.index
)

similarity_df.head()

MovieID,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
MovieID,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.333333,0.408248,0.408248,0.577350,0.0,0.408248,0.408248,0.0,0.000000,...,0.577350,0.408248,0.666667,0.000000,0.0,0.577350,0.000000,0.000000,0.000000,0.0
2,0.333333,1.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.816497,0.0,0.333333,...,0.000000,0.000000,0.666667,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
3,0.408248,0.000000,1.000000,0.500000,0.707107,0.0,1.000000,0.000000,0.0,0.000000,...,0.707107,0.500000,0.000000,0.000000,0.0,0.707107,0.000000,0.000000,0.000000,0.0
4,0.408248,0.000000,0.500000,1.000000,0.707107,0.0,0.500000,0.000000,0.0,0.000000,...,0.707107,1.000000,0.000000,0.408248,0.0,0.707107,0.707107,0.707107,0.707107,0.5
5,0.577350,0.000000,0.707107,0.707107,1.000000,0.0,0.707107,0.000000,0.0,0.000000,...,1.000000,0.707107,0.000000,0.000000,0.0,1.000000,0.000000,0.000000,0.000000,0.0


In [6]:
movie_id = 1

similar_movies = similarity_df[movie_id].sort_values(
    ascending=False
)

similar_movies.head(10)

MovieID
3114    1.0
3754    1.0
3611    1.0
1064    1.0
3751    1.0
2141    1.0
2354    1.0
1       1.0
2355    1.0
2142    1.0
Name: 1, dtype: float64

In [7]:
def get_recommendations(movie_title, top_n=10):

    # Find the selected movie
    matching_movies = movies[
        movies["Title"].str.casefold() == movie_title.casefold()
    ]

    if matching_movies.empty:
        return "Movie not found. Please check the movie title."

    # Get its MovieID
    movie_id = matching_movies.iloc[0]["MovieID"]

    # Retrieve similarity scores
    scores = similarity_df[movie_id]

    # Remove the selected movie itself
    scores = scores.drop(movie_id)

    # Select the most similar movies
    top_movies = scores.sort_values(
        ascending=False
    ).head(top_n)

    # Retrieve movie information
    recommendations = movie_details.loc[
        top_movies.index,
        ["Title", "Genres"]
    ].copy()

    # Add similarity scores
    recommendations["SimilarityScore"] = top_movies

    return recommendations.reset_index()

In [8]:
get_recommendations("Toy Story (1995)")

,MovieID,Title,Genres,SimilarityScore
0,2354,"Rugrats Movie, The (1998)",Animation|Children's|Comedy,1.000000
1,2355,"Bug's Life, A (1998)",Animation|Children's|Comedy,1.000000
2,1064,Aladdin and the King of Thieves (1996),Animation|Children's|Comedy,1.000000
3,2141,"American Tail, An (1986)",Animation|Children's|Comedy,1.000000
4,3114,Toy Story 2 (1999),Animation|Children's|Comedy,1.000000
5,3611,Saludos Amigos (1943),Animation|Children's|Comedy,1.000000
6,2142,"American Tail: Fievel Goes West, An (1991)",Animation|Children's|Comedy,1.000000
7,3751,Chicken Run (2000),Animation|Children's|Comedy,1.000000
8,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Animation|Children's|Comedy,1.000000
9,2078,"Jungle Book, The (1967)",Animation|Children's|Comedy|Musical,0.866025


In [9]:
get_recommendations("Jumanji (1995)")

,MovieID,Title,Genres,SimilarityScore
0,2005,"Goonies, The (1985)",Adventure|Children's|Fantasy,1.0
1,126,"NeverEnding Story III, The (1994)",Adventure|Children's|Fantasy,1.0
2,2162,"NeverEnding Story II: The Next Chapter, The (1...",Adventure|Children's|Fantasy,1.0
3,2161,"NeverEnding Story, The (1984)",Adventure|Children's|Fantasy,1.0
4,2043,Darby O'Gill and the Little People (1959),Adventure|Children's|Fantasy,1.0
5,1009,Escape to Witch Mountain (1975),Adventure|Children's|Fantasy,1.0
6,2399,Santa Claus: The Movie (1985),Adventure|Children's|Fantasy,1.0
7,60,"Indian in the Cupboard, The (1995)",Adventure|Children's|Fantasy,1.0
8,1967,Labyrinth (1986),Adventure|Children's|Fantasy,1.0
9,56,Kids of the Round Table (1995),Adventure|Children's|Fantasy,1.0


In [10]:
get_recommendations("Toy Story (1995)", top_n=5)

,MovieID,Title,Genres,SimilarityScore
0,2354,"Rugrats Movie, The (1998)",Animation|Children's|Comedy,1.0
1,2355,"Bug's Life, A (1998)",Animation|Children's|Comedy,1.0
2,1064,Aladdin and the King of Thieves (1996),Animation|Children's|Comedy,1.0
3,2141,"American Tail, An (1986)",Animation|Children's|Comedy,1.0
4,3114,Toy Story 2 (1999),Animation|Children's|Comedy,1.0


In [11]:
get_recommendations("This Movie Does Not Exist")

'Movie not found. Please check the movie title.'